# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    load_all_recordings, FT_SNAP_HZ, build_settings_dataframe,
    # Offline filtering investigation
    get_trial_raw_channels_window, apply_emg_filter, differential_bandpass_filter,
    get_trial_window_offline, get_trial_window_auto, make_filtering_viewer,
    find_digin_onset_in_emg_blocks,
    get_trial_raw_channels_window_oe,
    build_session_emg_filter_cache, get_trial_window_from_session_cache,
    FILTERING_PROTOCOL_OFFLINE, FILTERING_PROTOCOL_ONLINE,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [2]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    
    # HRPilot-23 Recordings

    
    # HRPilot-25 Recordings
    #("HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB1_HRPILOT-25_BOOTH2_250US_10KHZ_8-25-26",        10000.0),
    #("HRPILOT-25 100US",  "Calibration/HRPilot-25/CALIB2_HRPILOT-25_BOOTH2_100US_10KHZ_9-2-26",        10000.0),
    #("Calib3 HRPILOT-25 250US",  "Calibration/HRPilot-25/CALIB3_HRPILOT-25_BOOTH1_250US_10KHZ_9-10-26",        10000.0),

        
    
    # HRPilot-26 Recordings
    #("HRPILOT-26 250US",  "Calibration/HRPilot-26/CALIB1_HRPILOT-26_BOOTH2_250US_10KHZ_8-27-26",        10000.0),
    #("HRPILOT-26 100US",  "Calibration/HRPilot-26/CALIB2_HRPILOT-26_BOOTH1_100US_10KHZ_9-2-26",        10000.0),
    
    # HRPilot-34 Recordings
    #("HRPILOT-34 250US",  "Calibration/HRPilot-34/CALIB1_HRPILOT-34_BOOTH2_250US_10KHZ_9-2-26",        10000.0),
    #("Calib2 HRPILOT-34 250US",  "Calibration/HRPilot-34/CALIB2_HRPILOT-34_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    #("CM1 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM1_HRPILOT-34_BOOTH1_250US_10KHZ_9-14-26",        10000.0),
    #("CM2 HRPILOT-34 250US",  "Calibration/HRPilot-34/CM2_HRPILOT-34_BOOTH1_250US_10KHZ_9-15-26",        10000.0),
    #("Calib3 HRPILOT-34 100US",  "Calibration/HRPilot-34/CALIB3_HRPILOT-34_BOOTH1_100US_10KHZ_9-15-26",        10000.0),
        
    
    # HRPilot-36 Recordings
    #("Calib2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CALIB2_HRPILOT-36_BOOTH2_250US_10KHZ_9-10-26",        10000.0),
    #("CM1 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM1_HRPILOT-36_BOOTH2_250US_10KHZ_9-14-26",        10000.0),
    #("CM2 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM2_HRPILOT-36_BOOTH2_250US_10KHZ_9-15-26",        10000.0),
    ("CM3 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
    #("CM3 FT OFfline HRPILOT-36 250US",  "Calibration/HRPilot-36/CM3_FT_OFFLINE_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26",        10000.0),
    ("CM4 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26",        10000.0),
    ("CM5 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM5_HRPILOT-36_BOOTH1_250US_10KHZ_9-21-26",        10000.0),
    ("CM6 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM6_HRPILOT-36_BOOTH1_250US_10KHZ_9-22-26",        10000.0),
    ("CM7 HRPILOT-36 250US",  "Calibration/HRPilot-36/CM7_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-23-26",        10000.0),
        
    
    #HRPilot-19 Recordings
    #("Calib1 HRPILOT-19 250US",  "Calibration/CALIB1_HRPILOT-19_BOOTH2_250US_10KHZ_9-18-26",        10000.0),
    
    #HRPilot-21 Recordings
    #("Calib1 HRPILOT-21 250US",  "Calibration/CALIB1_HRPILOT-21_BOOTH1_250US_10KHZ_9-18-26",        10000.0),
        
    

    
    
    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

5 recording(s) configured.
  [0] 'CM3 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM3_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26  (sample_rate=10000.0 Hz)
  [1] 'CM4 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26  (sample_rate=10000.0 Hz)
  [2] 'CM5 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM5_HRPILOT-36_BOOTH1_250US_10KHZ_9-21-26  (sample_rate=10000.0 Hz)
  [3] 'CM6 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM6_HRPILOT-36_BOOTH1_250US_10KHZ_9-22-26  (sample_rate=10000.0 Hz)
  [4] 'CM7 HRPILOT-36 250US'  →  Calibration/HRPilot-36/CM7_FILTFILT_HRPILOT-36_BOOTH2_250US_10KHZ_9-23-26  (sample_rate=10000.0 Hz)


In [3]:
# ── Load all recordings (auto-detects V2 / V3) ──────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

_all_recordings = load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM)
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'CM3 HRPILOT-36 250US'  (Calibration/HRPilot-36/CM3_HRPILOT-36_BOOTH2_250US_10KHZ_9-17-26)
   .hrs1: not found
   .hrs2: 804 trials  (Control Mode)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V3  |  Stages: ['control_mode']  |  SR: 10000.0 Hz
   Settings sidecars: 1/1 stage(s)

── Loading: 'CM4 HRPILOT-36 250US'  (Calibration/HRPilot-36/CM4_HRPILOT-36_BOOTH1_250US_10KHZ_9-18-26)
   .hrs1: not found
   .hrs2: 403 trials  (Control Mode)
   .hrft: 165 trials across 1 file(s)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   .hrft Hz values (snapped): ['Single Pulse', '5.0 Hz', '10.0 Hz']
   [OFFLINE] CM4 HRPILOT-36 250US/Control Mode (.hrs2): 0/403 trials rebuilt (raw→diff→filtfilt)  (403 kept stored — xcorr miss)
   App V3  |  Stages: ['control_mode', 'frequency_test']  |  SR: 10000.0 Hz
   Settings sidecars: 2/2 stage(s)

── Loading: 'CM5 HRPILOT-36 250US'  (Calibration/HRPilot-36/CM5_HRPILOT-36_BOOTH1_25

# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [4]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'CM3 HRPILOT-36 250US'
  Control Mode (.hrs2): 804 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [5]:
# ── Recording & Stage Viewer Factory ───────────────────────────────────────────
# Each viewer section below has its own independent Recording + Stage dropdowns.
# Set each viewer to a different recording/stage to compare them side by side.
# make_viewer() is defined in helpers.py
from IPython.display import display as _disp

# ── Loaded recordings summary ────────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()
print('Each viewer below has its own Recording + Stage dropdowns for independent selection.')


'''
# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()
'''

5 recording(s) loaded:
  'CM3 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 804 trials
  'CM4 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 403 trials
    · Frequency Test (.hrft): 165 trials
  'CM5 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 831 trials
  'CM6 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 808 trials
  'CM7 HRPILOT-36 250US'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 803 trials

Each viewer below has its own Recording + Stage dropdowns for independent selection.


"\n# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────\ndef _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):\n    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')\n    plot_amplitude_distribution(trials, header)\n\n_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)\n_disp(_hist_widget)\n_hist_render()\n"

# Section 3b: Offline Filtering Investigation

For recordings tagged `filtering_protocol == "OFFLINE"`, the two raw EMG electrode channels (pre-differential-subtraction, pre-bandpass) are stored in the continuous background stream and can be reprocessed here with any filtering approach.

Pick a Recording / Stage / Trial below, then freely toggle:
- **Mode**: raw (unfiltered differential) vs. band-pass
- **Method**: `lfilter` (causal, matches this codebase's historical/online-style filtering) vs. `filtfilt` (zero-phase)
- **Low/High cutoff and filter order**
- **Notch filter** on/off (mains hum), with frequency and Q

Four quick-preset buttons cover the standard investigations: raw unfiltered; the existing 100–1000 Hz order-2 band with `filtfilt`; a broad 3–5000 Hz low-order (3) band with no notch; and that same broad band with the notch filter added back in.

The reprocessed signal is plotted against the app's own stored `trial_data` (toggle-able) for direct comparison, with M/H window shading and live M/H size numbers in the title. Recordings not tagged OFFLINE show a warning — the "raw" channels there may already have been filtered/differenced upstream by Open Ephys.

In [ ]:
# ── Offline Filtering Investigation: Interactive Viewer ────────────────────────
# Self-contained widget: Recording/Stage/Trial dropdowns + filter controls.
# Only meaningful for OFFLINE recordings — for ONLINE, raw channels may already
# be filtered/differenced upstream by Open Ephys (warning shown in the viewer).

_filt_widget, _filt_render = make_filtering_viewer(
    _all_recordings, _active_rec_label,
    pre_ms=5.0, post_ms=25.0,
    m_start_ms=2.2, m_end_ms=4.2,
    h_start_ms=6.0, h_end_ms=9.6,
)
display(_filt_widget)
_filt_render()

# Section 6: HRS2 Detailed Analysis

Interactive averaged-waveform paged grid (with M/H peak markers and signal overlays) plus the normalized and raw recruitment curves. The cell below sets the analysis parameters; the cell after that calls `plot_hrs2_analysis`.

In [7]:
#  Configuration 
PRE_PLOT_MS  = 2   # ms before stim onset to display
POST_PLOT_MS = 15  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
#M_WAVE_START_MS = 3
#M_WAVE_END_MS= 5.5
#H_WAVE_START_MS = 9.0
#H_WAVE_END_MS   = 12

M_WAVE_START_MS = 2.2 
M_WAVE_END_MS= 4.2
H_WAVE_START_MS = 6.5   
H_WAVE_END_MS   = 11.5

PRE_AVG_MS  = 2   # ms before stim onset
POST_AVG_MS = 15  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [8]:
# ── Pre-compute Comparison Data ───────────────────────────────────────────────────────
# Computed once here (after configuration constants are set) for instant rendering
# in the comparison plot below. Re-run this cell if you change PRE_AVG_MS,
# POST_AVG_MS, M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, or H_WAVE_END_MS.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'Comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

Comparison data pre-computed for 5 recording(s), 2 stage(s): ['control_mode', 'frequency_test']


In [9]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = True   # True → collapse every amplitude into one group
MERGED_GROUPS = []
#[(0,0.108), (0.108,0.23), (0.23,0.43), (0.43,0.50), (0.50,0.60), (0.60,0.70)]

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = True  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [10]:
rec = _all_recordings[_active_rec_label]
print(rec.get("filtering_protocol"))

None


In [15]:
# ── FILTER_CONFIG: choose the signal pipeline used in this analysis cell ───────
#
# 'default'            → app's own stored trial_data (100–1000 Hz sosfilt, causal).
# 'bp_100_1000_ff'     → 100–1000 Hz bandpass, filtfilt (zero-phase).
# 'raw_diff'           → ch_b − ch_a only, NO bandpass.
# 'bp_3_5000_ff'       → 3–5000 Hz bandpass, order 3, filtfilt (zero-phase).
# 'bp_3_5000_ff_notch' → 3–5000 Hz bandpass, order 3, filtfilt + 60 Hz notch.
#
# Onset alignment (all non-default configs):
#   Cross-correlates the session-long filtered signal against trial.trial_data to
#   find the exact onset — same approach as reconstruct_raw_peristimulus.py,
#   verified 0.0000% error.  Wall-clock proximity (trigger_wall_time_ms) narrows
#   the search window; blk.ts_background_emitted orders the blocks chronologically.
#   ts_open_ephys_sent is NOT used — it carries wall-clock ms, not OE sample numbers.
FILTER_CONFIG   = 'bp_100_1000_ff'  # choose one of the above configs
ONSET_OFFSET_MS = 0.0   # kept for API compatibility; not used

import copy as _copy

# Session filter cache — persists across renders; cleared when this cell is re-run.
_SESSION_CACHES = {}

def _apply_filter_config(trials, emg_blocks, sr, config, onset_offset_ms=0.0):
    """Return trials with trial_data replaced by the chosen offline filter config.

    All non-default configs:
      1. Build session cache once (sosfilt 100-1000 Hz, used for xcorr alignment).
      2. Slice raw_diff from the cache with a 300 ms warmup prepended.
      3. Apply the config's filter to the extended window.
      4. Trim the warmup and store the result in tc.trial_data.
      5. Re-slice stim_adc_data to the same (pre_ms, post_ms) window so
         plot_hrs2_analysis can index it correctly with the new onset_sample_index.
    """
    if config == 'default':
        return trials

    _pre_ms      = PRE_AVG_MS
    _post_ms     = POST_AVG_MS
    _pre_samp    = int(round(_pre_ms  * sr / 1000.0))
    _post_samp   = int(round(_post_ms * sr / 1000.0))
    _WARMUP_MS   = 300.0
    _warmup_samp = int(round(_WARMUP_MS * sr / 1000.0))

    # Build session cache once per stage (needed for xcorr alignment in all configs).
    _cache_key = id(emg_blocks[0]) if emg_blocks else 0
    if _cache_key not in _SESSION_CACHES:
        print('Building session filter cache (one-time per stage)...')
        _SESSION_CACHES[_cache_key] = build_session_emg_filter_cache(emg_blocks, sr)
    sc = _SESSION_CACHES.get(_cache_key)
    if sc is None:
        print('Session filter cache could not be built — returning stored signal.')
        return trials

    result = []
    for t in trials:
        # Slice raw_diff with warmup prepended so any IIR filter has history to settle.
        win = get_trial_window_from_session_cache(
            t, sc, _pre_ms + _WARMUP_MS, _post_ms, use_raw=True)
        if win is None:
            result.append(t)
            continue
        _t_ms_full, sig_full = win

        if config == 'raw_diff':
            sig = sig_full[_warmup_samp:]
        elif config == 'bp_100_1000_ff':
            sig = apply_emg_filter(sig_full, sr, mode='bandpass',
                                   lowcut=100.0, highcut=1000.0, order=2,
                                   method='filtfilt')[_warmup_samp:]
        elif config == 'bp_3_5000_ff':
            sig = apply_emg_filter(sig_full, sr, mode='bandpass',
                                   lowcut=3.0, highcut=5000.0, order=2,
                                   method='filtfilt')[_warmup_samp:]
        elif config == 'bp_3_5000_ff_notch':
            sig = apply_emg_filter(sig_full, sr, mode='bandpass',
                                   lowcut=3.0, highcut=5000.0, order=2,
                                   method='filtfilt', notch=True,
                                   notch_freq=60.0, notch_q=30.0)[_warmup_samp:]
        else:
            result.append(t)
            continue

        tc = _copy.copy(t)
        tc.trial_data = sig
        tc.onset_sample_index = _pre_samp

        # Re-slice stim_adc_data to the same (pre_ms → post_ms) window so that
        # its onset is also at _pre_samp — consistent with tc.onset_sample_index.
        # Without this, plot_hrs2_analysis would slice the original array using
        # the new _pre_samp index, landing at the wrong position in the signal.
        _orig_osi = getattr(t, 'onset_sample_index', -1)
        if _orig_osi is not None and _orig_osi >= 0:
            _sa = np.asarray(getattr(t, 'stim_adc_data', []), dtype=float)
            if len(_sa) > 0:
                _sa_s0 = _orig_osi - _pre_samp
                _sa_s1 = _orig_osi + _post_samp
                if 0 <= _sa_s0 and _sa_s1 <= len(_sa):
                    tc.stim_adc_data = _sa[_sa_s0:_sa_s1]

        result.append(tc)

    return result


# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from IPython.display import display as _disp

def _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _sr = sr or h1h.sample_rate
    trials_filt = _apply_filter_config(trials, emg_blocks, _sr, FILTER_CONFIG, ONSET_OFFSET_MS)
    tp = _apply_merge(trials_filt)
    _flbl = '' if FILTER_CONFIG == 'default' else f'  [filter: {FILTER_CONFIG}]'
    print(f'\n── Analysis: {stage_label}  ({len(trials)} trials)  [{rec_label}]{_flbl}')
    plot_hrs2_analysis(
        tp, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=_sr,
        emg_blocks=emg_blocks,
    )

_ana_widget, _ana_render = make_viewer(_all_recordings, _active_rec_label, _render_ana)
_disp(_ana_widget)
_ana_render()

# Section 6c: Frequency Test Analysis (V3 FT)

Three views available via the **View** toggle:

- **H/M MRA Per Pulse** — mean ± 1σ H-wave and M-wave MRA at each pulse position across all trials. Uses `pulse_h_wave_mra` / `pulse_m_wave_mra` pre-stored by the app (mean rectified average within each window). The most-recent trial is overlaid as a dashed line. Shows homosynaptic (rate-dependent) depression across the train.
- **Avg Waveforms** — paged 2×3 grid. Each tile is one pulse position: individual trial segments are drawn at low alpha, the bold black trace is the cross-trial mean. M-wave window is blue-shaded; H-wave window is green-shaded (matching the HRS2 Analysis viewer). MRA annotations are boxed above each window. A colorbar at the top encodes pulse # (blue = pulse 1, red = last). Use the **Page** slider to page through pulses.
- **H/M Peak Per Pulse** — same structure as H/M MRA Per Pulse, but computes `max(|EMG|)` within each wave window directly from the raw EMG trace, then averages across trials.

M/H wave windows are shared with the HRS2 configuration constants (`M_WAVE_START_MS`, `H_WAVE_START_MS`, etc.).

In [16]:
# ── Frequency Test Analysis: Interactive Viewer ────────────────────────────────
# View toggle:  H/M MRA Per Pulse | Avg Waveforms | H/M Peak Per Pulse
# Page slider (Avg Waveforms only): step through pulse positions 6 at a time.
# M/H wave windows use M_WAVE_START_MS / H_WAVE_START_MS from the config cell above.
from ipywidgets import Dropdown, ToggleButtons, IntSlider, Output, VBox
from IPython.display import display as _disp

_ft_recs = [rl for rl in _all_recordings if _all_recordings[rl].get('ft_trials')]
if not _ft_recs:
    print("No Frequency Test data loaded (.hrft not found in any recording directory).")
else:
    _ft_rec_d = Dropdown(
        options=_ft_recs, value=_ft_recs[0],
        description='Recording:', layout={'width': '600px'}
    )
    _ft_view_d = ToggleButtons(
        options=[
            ('H/M MRA Per Pulse', 'depression'),
            ('Avg Waveforms',     'waveforms'),
            ('H/M Peak Per Pulse','peak'),
        ],
        description='View:', style={'button_width': '185px'},
    )
    # Page slider — shown only for the Avg Waveforms view
    _ft_page_s = IntSlider(
        min=1, max=1, step=1, value=1,
        description='Page:', layout={'width': '450px', 'visibility': 'hidden'}
    )
    _ft_out = Output()

    def _ft_render():
        rl   = _ft_rec_d.value
        vw   = _ft_view_d.value
        rec  = _all_recordings[rl]
        ft_t = rec.get('ft_trials', [])
        ft_h = rec.get('ft_header')
        sr   = rec.get('sample_rate') or getattr(ft_h, 'sample_rate', None)
        if not ft_t or ft_h is None:
            return

        # Update page slider max to match actual pulse count / page size
        n_p  = getattr(ft_h, 'n_pulses_per_train', 0) or \
               max((len(getattr(t, 'pulse_h_wave_mra', [])) for t in ft_t), default=1)
        tot  = max(1, int(np.ceil(n_p / N_PER_PAGE)))
        _ft_page_s.max = tot
        _ft_page_s.layout.visibility = 'visible' if vw == 'waveforms' else 'hidden'

        hz = round(1e6 / ft_h.event_period_us, 1) if getattr(ft_h, 'event_period_us', 0) else '?'
        with _ft_out:
            _ft_out.clear_output(wait=True)
            print(f'Frequency Test: {len(ft_t)} trials  |  '
                  f'{n_p} pulses/train  |  '
                  f'{hz} Hz  [{rl}]')
            if vw == 'depression':
                plot_ft_depression_curve(ft_t, ft_h, sample_rate=sr)
            elif vw == 'waveforms':
                plot_ft_averaged_waveforms(
                    ft_t, ft_h,
                    pre_pulse_ms=2.0, post_pulse_ms=POST_PLOT_MS,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                    sample_rate=sr, n_per_page=N_PER_PAGE,
                    page=_ft_page_s.value - 1,
                )
            else:  # peak
                plot_ft_peak_curve(
                    ft_t, ft_h, sample_rate=sr,
                    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
                    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
                )

    def _ft_on_rec(c):
        ft_t = _all_recordings[_ft_rec_d.value].get('ft_trials', [])
        _ft_page_s.value = 1
        _ft_render()

    def _ft_on_page(c):
        if _ft_view_d.value == 'waveforms':
            _ft_render()

    _ft_rec_d.observe(_ft_on_rec,                   names='value')
    _ft_view_d.observe(lambda c: _ft_render(),       names='value')
    _ft_page_s.observe(_ft_on_page,                  names='value')

    _disp(VBox([_ft_rec_d, _ft_view_d, _ft_page_s, _ft_out]))
    _ft_render()

# Section 6b: H-Reflex Size Across Recordings

**H-reflex size per amplitude group** is the **Mean Rectified Amplitude (MRA) of the averaged bipolar waveform** in the H-wave window, minus the MRA of the pre-stimulus background:

> **size (µV) = mean|avg_bip(t ∈ [H_START, H_END])| − mean|avg_bip(t < 0)|**

Steps per amplitude group in each recording:
1. All trials at that amplitude are time-aligned and averaged → `avg_bip`
2. **H-wave MRA** = mean of |avg_bip| within [H_WAVE_START_MS, H_WAVE_END_MS]
3. **Background MRA** = mean of |avg_bip| in the pre-stimulus window (t < 0)
4. **H-reflex size** = H-wave MRA − background MRA

This matches the green `H: X.X µV` annotation shown in the HRS2 Analysis viewer above, with background subtracted.

The plot below shows one **box-and-whisker per recording** in `RECORDING_DIRS` order, with each amplitude group contributing one data point:
- **Box** — interquartile range (25th–75th percentile across amplitude groups)
- **Horizontal bar** — median
- **◆ diamond** — mean
- **Whiskers** — mean ± 1 standard deviation
- **n=** — number of amplitude groups in that recording for the selected stage
- **Dashed line** — connects per-recording means in `RECORDING_DIRS` order

In [17]:
# ── Cross-Recording Comparison Plot ──────────────────────────────────────────────────
# Toggle between H-Reflex size, M-Wave size, and Background MRA (pre-stim EMG level).
# Re-run the pre-compute cell above if you change analysis configuration constants.
from ipywidgets import Dropdown, ToggleButtons, Output, VBox
from IPython.display import display as _disp

_xr_stage_d = Dropdown(
    options=[(_slbl, _sk) for _sk, _slbl in _xr_stages.items()],
    description='Stage:', layout={'width': '480px'}
)
if _xr_stages:
    _xr_stage_d.value = next(iter(_xr_stages))

_xr_metric_d = ToggleButtons(
    options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'), ('Background MRA', 'background')],
    description='Metric:',
    style={'button_width': '160px'},
)

_xr_out = Output()

def _xr_render():
    sk = _xr_stage_d.value
    mt = _xr_metric_d.value
    if not sk:
        return
    with _xr_out:
        _xr_out.clear_output(wait=True)
        plot_h_reflex_comparison(_xr_cache, RECORDING_DIRS, sk, _xr_stages, metric=mt)

_xr_stage_d.observe(lambda c: _xr_render(), names='value')
_xr_metric_d.observe(lambda c: _xr_render(), names='value')
_disp(VBox([_xr_stage_d, _xr_metric_d, _xr_out]))
_xr_render()